In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from utils.tobii import get_anotacions

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
PROJECT = "../data/tobii/"
general = (
    pd.read_csv(PROJECT + "general.tsv", sep="\t")
    .drop(["Event", "Recording timestamp", "Computer timestamp"], axis=1)
    .drop_duplicates()
)
general[["Participant name", "sexe"]]

In [ ]:
anotacions = get_anotacions(PROJECT + "anotacions.tsv")
anotacions.head()

In [ ]:
mused_all = pd.read_csv("../data/mused_all_clean.csv")
mused_chosen = pd.read_csv("../data/mused_chosen_data.csv")
mused = mused_all[mused_all.id.isin(mused_chosen.id)].copy()
mused["num_id"] = mused.id.str.split("_").str[1].astype(int)

# Merge participant annotations with gold standard
df = anotacions.merge(
    mused[["num_id", "sexist"]].rename(columns={"num_id": "text", "sexist": "gold_sexist"}),
    on="text",
    how="inner",
)
df["correct"] = (df["sexist"] == df["gold_sexist"]).astype(int)
print(f"Merged dataset: {df.shape[0]} rows ({df.user.nunique()} participants x {df.text.nunique()} texts)")
df.head(10)

## 1. Per-participant accuracy vs gold standard

In [ ]:
def compute_metrics(group):
    tp = ((group["sexist"] == 1) & (group["gold_sexist"] == 1)).sum()
    tn = ((group["sexist"] == 0) & (group["gold_sexist"] == 0)).sum()
    fp = ((group["sexist"] == 1) & (group["gold_sexist"] == 0)).sum()
    fn = ((group["sexist"] == 0) & (group["gold_sexist"] == 1)).sum()
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return pd.Series(
        {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "n_texts": len(group),
        }
    )


participant_metrics = df.groupby("user").apply(compute_metrics).reset_index()
participant_metrics.sort_values("f1", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 scores
pm = participant_metrics.sort_values("f1", ascending=False)
colors = ["#2ecc71" if v >= 0.7 else "#f39c12" if v >= 0.5 else "#e74c3c" for v in pm["f1"]]
axes[0].barh(pm["user"], pm["f1"], color=colors)
axes[0].set_xlabel("F1 Score")
axes[0].set_title("F1 Score per Participant (vs Gold Standard)")
axes[0].set_xlim(0, 1)
axes[0].axvline(x=0.5, color="gray", linestyle="--", alpha=0.5)

# Confusion matrix components
pm_sorted = pm.sort_values("user")
x = np.arange(len(pm_sorted))
width = 0.2
axes[1].bar(x - 1.5 * width, pm_sorted["tp"], width, label="TP", color="#2ecc71")
axes[1].bar(x - 0.5 * width, pm_sorted["tn"], width, label="TN", color="#3498db")
axes[1].bar(x + 0.5 * width, pm_sorted["fp"], width, label="FP", color="#e74c3c")
axes[1].bar(x + 1.5 * width, pm_sorted["fn"], width, label="FN", color="#f39c12")
axes[1].set_xticks(x)
axes[1].set_xticklabels(pm_sorted["user"], rotation=45, ha="right")
axes[1].set_ylabel("Count")
axes[1].set_title("Confusion Matrix Components per Participant")
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Inter-rater agreement

In [ ]:
def cohens_kappa(r1, r2):
    """Compute Cohen's kappa between two binary raters."""
    r1, r2 = np.asarray(r1), np.asarray(r2)
    n = len(r1)
    p_o = (r1 == r2).mean()  # observed agreement
    p_11 = ((r1 == 1) & (r2 == 1)).mean()
    p_00 = ((r1 == 0) & (r2 == 0)).mean()
    p_10 = ((r1 == 1) & (r2 == 0)).mean()
    p_01 = ((r1 == 0) & (r2 == 1)).mean()
    p_e = (r1 == 1).mean() * (r2 == 1).mean() + (r1 == 0).mean() * (r2 == 0).mean()
    if p_e == 1:
        return 1.0
    return (p_o - p_e) / (1 - p_e)


def fleiss_kappa(ratings_matrix):
    """Compute Fleiss' kappa for multiple raters.
    ratings_matrix: shape (n_items, n_categories) with counts per category.
    """
    n_items, n_categories = ratings_matrix.shape
    n_raters = ratings_matrix.sum(axis=1)[0]  # assume equal raters per item
    N = n_items
    K = n_categories
    P = ratings_matrix / n_raters  # proportion of assignments per category per item
    P_i = (P * (P - 1)).sum(axis=1).sum() / (N * K * (K - 1) / (K))
    p_j = ratings_matrix.sum(axis=0) / (N * n_raters)
    P_e = (p_j**2).sum()
    if P_e == 1:
        return 1.0
    return (P_i - P_e) / (1 - P_e)


# Pairwise Cohen's kappa
users = sorted(df.user.unique())
kappa_matrix = pd.DataFrame(np.nan, index=users, columns=users)

for u1, u2 in combinations(users, 2):
    r1 = df[df.user == u1].set_index("text")["sexist"]
    r2 = df[df.user == u2].set_index("text")["sexist"]
    common = r1.index.intersection(r2.index)
    k = cohens_kappa(r1.loc[common], r2.loc[common])
    kappa_matrix.loc[u1, u2] = k
    kappa_matrix.loc[u2, u1] = k

for u in users:
    kappa_matrix.loc[u, u] = 1.0

print("Pairwise Cohen's Kappa:")
kappa_matrix.round(3)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    kappa_matrix.astype(float),
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    center=0.5,
    vmin=-0.1,
    vmax=1.0,
    square=True,
    linewidths=0.5,
)
plt.title("Pairwise Cohen's Kappa between Participants")
plt.tight_layout()
plt.show()

In [ ]:
# Fleiss' kappa across all participants
pivot = df.pivot_table(index="text", columns="user", values="sexist")
n_items = len(pivot)
n_categories = 2
n_raters = len(users)

ratings_matrix = np.zeros((n_items, n_categories), dtype=int)
for cat in range(n_categories):
    ratings_matrix[:, cat] = (pivot == cat).sum(axis=1).values

fk = fleiss_kappa(ratings_matrix)
print(f"Fleiss' Kappa (all participants): {fk:.3f}")

# Also compute vs gold standard
gold_kappas = {}
for u in users:
    r1 = df[df.user == u].set_index("text")["sexist"]
    r2 = df[df.user == u].set_index("text")["gold_sexist"]
    common = r1.index.intersection(r2.index)
    gold_kappas[u] = cohens_kappa(r1.loc[common], r2.loc[common])

gold_kappa_df = pd.DataFrame.from_dict(gold_kappas, orient="index", columns=["kappa_vs_gold"])
print("\nCohen's Kappa vs Gold Standard:")
gold_kappa_df.sort_values("kappa_vs_gold", ascending=False)

## 3. Confidence analysis

In [ ]:
# Accuracy by confidence level
conf_metrics = (
    df.groupby("confidence")
    .apply(
        lambda g: pd.Series(
            {
                "accuracy": g["correct"].mean(),
                "count": len(g),
                "n_correct": g["correct"].sum(),
                "n_wrong": (~g["correct"].astype(bool)).sum(),
                "pct_of_total": len(g) / len(df) * 100,
            }
        )
    )
    .reset_index()
)
conf_metrics["accuracy"] = conf_metrics["accuracy"].round(3)
print("Accuracy by confidence level:")
conf_metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy by confidence
conf_sorted = conf_metrics.sort_values("confidence")
bars = axes[0].bar(
    conf_sorted["confidence"].astype(str), conf_sorted["accuracy"], color=["#e74c3c", "#f39c12", "#2ecc71"]
)
axes[0].set_xlabel("Confidence Level")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy by Confidence Level")
axes[0].set_ylim(0, 1)
for bar, acc in zip(bars, conf_sorted["accuracy"]):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{acc:.2f}", ha="center", fontweight="bold"
    )

# Distribution of confidence for correct vs incorrect
correct_conf = df[df["correct"] == 1]["confidence"]
wrong_conf = df[df["correct"] == 0]["confidence"]
bins = [0.5, 1.5, 2.5, 3.5]
axes[1].hist(correct_conf, bins=bins, alpha=0.7, label="Correct", color="#2ecc71", density=True)
axes[1].hist(wrong_conf, bins=bins, alpha=0.7, label="Incorrect", color="#e74c3c", density=True)
axes[1].set_xlabel("Confidence Level")
axes[1].set_ylabel("Density")
axes[1].set_title("Confidence Distribution: Correct vs Incorrect")
axes[1].legend()
axes[1].set_xticks([1, 2, 3])

# Per-participant confidence vs accuracy
user_conf = df.groupby(["user", "confidence"]).apply(lambda g: g["correct"].mean()).reset_index(name="accuracy")
user_conf_pivot = user_conf.pivot(index="user", columns="confidence", values="accuracy")
user_conf_pivot.plot(kind="bar", ax=axes[2], colormap="RdYlGn")
axes[2].set_xlabel("Participant")
axes[2].set_ylabel("Accuracy")
axes[2].set_title("Accuracy by Confidence per Participant")
axes[2].legend(title="Confidence")
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# High-confidence errors: texts where confidence=3 but wrong
high_conf_errors = df[(df["confidence"] == 3) & (df["correct"] == 0)].copy()
print(
    f"High-confidence errors: {len(high_conf_errors)} out of {len(df[df.confidence == 3])} high-confidence annotations ({len(high_conf_errors)/len(df[df.confidence == 3])*100:.1f}%)"
)
print()
if len(high_conf_errors) > 0:
    print("High-confidence errors by text and participant:")
    high_conf_errors[["user", "text", "sexist", "gold_sexist"]].sort_values(["text", "user"])

## 4. Per-text difficulty analysis

In [ ]:
# Per-text agreement with gold standard
text_analysis = (
    df.groupby("text")
    .apply(
        lambda g: pd.Series(
            {
                "gold_sexist": g["gold_sexist"].iloc[0],
                "pct_agree_with_gold": g["correct"].mean(),
                "n_agree": g["correct"].sum(),
                "n_disagree": (~g["correct"].astype(bool)).sum(),
                "n_participants": len(g),
                "mean_confidence": g["confidence"].mean(),
                "n_high_conf_wrong": ((g["confidence"] == 3) & (g["correct"] == 0)).sum(),
            }
        )
    )
    .reset_index()
)
text_analysis["pct_agree_with_gold"] = text_analysis["pct_agree_with_gold"].round(3)
text_analysis.sort_values("pct_agree_with_gold")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Difficulty ranking
ta = text_analysis.sort_values("pct_agree_with_gold")
colors = ["#e74c3c" if v < 0.5 else "#f39c12" if v < 0.7 else "#2ecc71" for v in ta["pct_agree_with_gold"]]
axes[0].barh(ta["text"].astype(str), ta["pct_agree_with_gold"], color=colors)
axes[0].set_xlabel("% Agreement with Gold Standard")
axes[0].set_title("Text Difficulty (lowest agreement = hardest)")
axes[0].axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="50%")
axes[0].axvline(x=0.7, color="gray", linestyle=":", alpha=0.5, label="70%")
axes[0].legend()

# Gold label distribution of hard vs easy texts
hard = text_analysis[text_analysis["pct_agree_with_gold"] < 0.6]
easy = text_analysis[text_analysis["pct_agree_with_gold"] >= 0.8]
med = text_analysis[(text_analysis["pct_agree_with_gold"] >= 0.6) & (text_analysis["pct_agree_with_gold"] < 0.8)]

categories = ["Hard (<60%)", "Medium (60-80%)", "Easy (>=80%)"]
sexist_counts = [
    (hard["gold_sexist"] == 1).sum(),
    (med["gold_sexist"] == 1).sum(),
    (easy["gold_sexist"] == 1).sum(),
]
non_sexist_counts = [
    (hard["gold_sexist"] == 0).sum(),
    (med["gold_sexist"] == 0).sum(),
    (easy["gold_sexist"] == 0).sum(),
]

x = np.arange(len(categories))
width = 0.35
axes[1].bar(x - width / 2, sexist_counts, width, label="Gold: Sexist", color="#e74c3c")
axes[1].bar(x + width / 2, non_sexist_counts, width, label="Gold: Not Sexist", color="#3498db")
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].set_ylabel("Number of texts")
axes[1].set_title("Gold Label Distribution by Difficulty")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Hardest texts: most disagreement
print("=== HARDEST TEXTS (lowest agreement with gold) ===")
hardest = text_analysis.nsmallest(10, "pct_agree_with_gold")
for _, row in hardest.iterrows():
    print(
        f"Text {int(row.text):>3d} | Gold: {'Sexist' if row.gold_sexist == 1 else 'Not Sexist':>10s} | "
        f"Agreement: {row.pct_agree_with_gold:.0%} ({int(row.n_agree)}/{int(row.n_participants)}) | "
        f"Mean conf: {row.mean_confidence:.1f} | High-conf errors: {int(row.n_high_conf_wrong)}"
    )

print("\n=== EASIEST TEXTS (highest agreement with gold) ===")
easiest = text_analysis.nlargest(10, "pct_agree_with_gold")
for _, row in easiest.iterrows():
    print(
        f"Text {int(row.text):>3d} | Gold: {'Sexist' if row.gold_sexist == 1 else 'Not Sexist':>10s} | "
        f"Agreement: {row.pct_agree_with_gold:.0%} ({int(row.n_agree)}/{int(row.n_participants)}) | "
        f"Mean conf: {row.mean_confidence:.1f} | High-conf errors: {int(row.n_high_conf_wrong)}"
    )

## 5. Gender-based analysis

In [ ]:
# Get gender from general metadata
gender_map = general.set_index("Participant name")["sexe"].to_dict()
print("Participant genders:")
for u in sorted(users):
    print(f"  {u}: {gender_map.get(u, 'unknown')}")

df["gender"] = df["user"].map(gender_map)

In [ ]:
gender_metrics = df.groupby("gender").apply(compute_metrics).reset_index()
print("Metrics by gender:")
gender_metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Overall accuracy by gender
gender_metrics_sorted = gender_metrics.sort_values("gender")
axes[0].bar(gender_metrics_sorted["gender"], gender_metrics_sorted["accuracy"], color=["#e74c3c", "#3498db"])
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy by Gender")
axes[0].set_ylim(0, 1)

# Confidence by gender
gender_conf = df.groupby(["gender", "confidence"]).apply(lambda g: g["correct"].mean()).reset_index(name="accuracy")
gender_conf_pivot = gender_conf.pivot(index="gender", columns="confidence", values="accuracy")
gender_conf_pivot.plot(kind="bar", ax=axes[1], colormap="RdYlGn")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy by Confidence & Gender")
axes[1].legend(title="Confidence")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=0)

# Error direction by gender: false positives vs false negatives
gender_errors = (
    df[df["correct"] == 0]
    .groupby("gender")
    .apply(
        lambda g: pd.Series(
            {
                "false_positive": ((g["sexist"] == 1) & (g["gold_sexist"] == 0)).sum(),
                "false_negative": ((g["sexist"] == 0) & (g["gold_sexist"] == 1)).sum(),
            }
        )
    )
    .reset_index()
)
gender_errors_sorted = gender_errors.sort_values("gender")
x = np.arange(len(gender_errors_sorted))
width = 0.35
axes[2].bar(
    x - width / 2,
    gender_errors_sorted["false_positive"],
    width,
    label="False Positive (called sexist, isn't)",
    color="#e74c3c",
)
axes[2].bar(
    x + width / 2,
    gender_errors_sorted["false_negative"],
    width,
    label="False Negative (missed sexist)",
    color="#f39c12",
)
axes[2].set_xticks(x)
axes[2].set_xticklabels(gender_errors_sorted["gender"])
axes[2].set_ylabel("Count")
axes[2].set_title("Error Types by Gender")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Per-text agreement by gender
text_gender = df.groupby(["text", "gender"]).apply(lambda g: g["correct"].mean()).reset_index(name="pct_agree")
text_gender_pivot = text_gender.pivot(index="text", columns="gender", values="pct_agree")
text_gender_pivot["diff"] = text_gender_pivot.iloc[:, 0] - text_gender_pivot.iloc[:, 1]
text_gender_pivot = text_gender_pivot.sort_values("diff")

print("Texts with largest gender disagreement:")
text_gender_pivot.head(10)

## 6. Multi-label sub-analysis

In [ ]:
# Sub-categories from gold standard
subcategories = [
    "irony",
    "humor",
    "implicit sexism",
    "stereotypes",
    "inequality",
    "discrimination",
    "objectification",
    "critique",
]

# Add subcategory columns to merged dataframe
mused_subs = mused[["num_id"] + subcategories].rename(columns={"num_id": "text"})
df_subs = df.merge(mused_subs, on="text", how="left")

print("Gold-standard sub-category prevalence (in sexist texts only):")
sexist_texts = df_subs[df_subs["gold_sexist"] == 1].drop_duplicates(subset="text")
for sub in subcategories:
    pct = sexist_texts[sub].mean() * 100
    print(f"  {sub:>20s}: {pct:.0f}% of sexist texts")

In [ ]:
# For each sub-category, compute participant accuracy on texts with that label
sub_analysis = []
for sub in subcategories:
    sub_texts = df_subs[df_subs[sub] > 0]["text"].unique()
    if len(sub_texts) == 0:
        continue
    sub_data = df_subs[df_subs["text"].isin(sub_texts)]
    sub_analysis.append(
        {
            "subcategory": sub,
            "n_texts": len(sub_texts),
            "accuracy": sub_data["correct"].mean(),
            "precision_on_sexist": (
                ((sub_data["sexist"] == 1) & (sub_data["gold_sexist"] == 1)).sum()
                / max((sub_data["sexist"] == 1).sum(), 1)
            ),
            "recall_on_sexist": (
                ((sub_data["sexist"] == 1) & (sub_data["gold_sexist"] == 1)).sum()
                / max((sub_data["gold_sexist"] == 1).sum(), 1)
            ),
        }
    )

sub_df = pd.DataFrame(sub_analysis).sort_values("accuracy")
sub_df["accuracy"] = sub_df["accuracy"].round(3)
sub_df["precision_on_sexist"] = sub_df["precision_on_sexist"].round(3)
sub_df["recall_on_sexist"] = sub_df["recall_on_sexist"].round(3)
print("Participant detection performance by gold-standard sub-category:")
sub_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sub-category accuracy
sub_sorted = sub_df.sort_values("accuracy")
colors = ["#e74c3c" if v < 0.5 else "#f39c12" if v < 0.7 else "#2ecc71" for v in sub_sorted["accuracy"]]
axes[0].barh(sub_sorted["subcategory"], sub_sorted["accuracy"], color=colors)
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Participant Accuracy by Sexism Sub-category")
axes[0].axvline(x=0.5, color="gray", linestyle="--", alpha=0.5)
axes[0].set_xlim(0, 1)

# Precision vs Recall scatter
axes[1].scatter(sub_df["precision_on_sexist"], sub_df["recall_on_sexist"], s=100, c="#2c3e50")
for _, row in sub_df.iterrows():
    axes[1].annotate(
        row["subcategory"],
        (row["precision_on_sexist"], row["recall_on_sexist"]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=9,
    )
axes[1].set_xlabel("Precision (sexist)")
axes[1].set_ylabel("Recall (sexist)")
axes[1].set_title("Precision vs Recall by Sub-category")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Error analysis: what do participants get wrong on sexist texts?
sexist_data = df_subs[df_subs["gold_sexist"] == 1].copy()

print("Error patterns on gold-standard sexist texts:")
print(f"  Total annotations on sexist texts: {len(sexist_data)}")
print(f"  Correct (detected sexist): {sexist_data['correct'].sum()} ({sexist_data['correct'].mean():.1%})")
print(
    f"  Missed (called not sexist): {(~sexist_data['correct'].astype(bool)).sum()} ({1 - sexist_data['correct'].mean():.1%})"
)

print("\nMissed sexist texts by sub-category presence:")
missed = sexist_data[sexist_data["correct"] == 0]
for sub in subcategories:
    texts_with_sub = sexist_data[sexist_data[sub] > 0]["text"].unique()
    missed_in_sub = missed[missed["text"].isin(texts_with_sub)]["text"].nunique()
    total_in_sub = len(texts_with_sub)
    if total_in_sub > 0:
        print(f"  {sub:>20s}: {missed_in_sub}/{total_in_sub} texts missed ({missed_in_sub/total_in_sub:.0%})")

## 7. Statistical Significance Tests

Given the small sample size (10 participants, 40 texts), we use non-parametric and exact tests.

In [ ]:
# Binomial test: is each participant's accuracy significantly above chance (50%)?
print("Binomial test: accuracy significantly > 50%? (one-sided)")
print(f"{'Participant':<12} {'Acc':>5} {'n':>3} {'p-value':>10} {'Sig?':>5}")
print("-" * 40)
for _, row in participant_metrics.iterrows():
    successes = int(row["tp"] + row["tn"])
    n = int(row["n_texts"])
    result = stats.binomtest(successes, n, p=0.5, alternative="greater")
    sig = "**" if result.pvalue < 0.01 else "*" if result.pvalue < 0.05 else "ns"
    print(f"{row['user']:<12} {row['accuracy']:5.1%} {n:3d} {result.pvalue:10.4f} {sig:>5}")

print(f"\n* p < .05, ** p < .01, ns = not significant")

In [ ]:
# Mann-Whitney U: do male vs female participants differ in per-text accuracy?
male_acc = df_subs[df_subs["gender"] == "home"].groupby("text")["correct"].mean()
female_acc = df_subs[df_subs["gender"] == "dona"].groupby("text")["correct"].mean()

# Align on shared texts
shared_texts = sorted(set(male_acc.index) & set(female_acc.index))
male_vals = male_acc.loc[shared_texts].values
female_vals = female_acc.loc[shared_texts].values

stat_u, p_u = stats.mannwhitneyu(male_vals, female_vals, alternative="two-sided")
print("Mann-Whitney U test: male vs female per-text accuracy")
print(f"  Male median: {np.median(male_vals):.3f}, Female median: {np.median(female_vals):.3f}")
print(f"  U = {stat_u:.1f}, p = {p_u:.4f} {'(significant)' if p_u < 0.05 else '(not significant)'}")

print()

# Fisher's exact test: are error counts different across genders?
sexist_texts = df_subs[df_subs["gold_sexist"] == 1]
male_errors_sexist = (sexist_texts[sexist_texts["gender"] == "home"]["correct"] == 0).sum()
male_total_sexist = (sexist_texts[sexist_texts["gender"] == "home"]["correct"] == 0).count()
female_errors_sexist = (sexist_texts[sexist_texts["gender"] == "dona"]["correct"] == 0).sum()
female_total_sexist = (sexist_texts[sexist_texts["gender"] == "dona"]["correct"] == 0).count()

table = np.array(
    [
        [male_errors_sexist, male_total_sexist - male_errors_sexist],
        [female_errors_sexist, female_total_sexist - female_errors_sexist],
    ]
)
odds, p_fisher = stats.fisher_exact(table)
print("Fisher's exact: error rate on sexist texts by gender")
print(f"  Male errors: {male_errors_sexist}/{male_total_sexist} ({male_errors_sexist/male_total_sexist:.1%})")
print(f"  Female errors: {female_errors_sexist}/{female_total_sexist} ({female_errors_sexist/female_total_sexist:.1%})")
print(f"  OR = {odds:.2f}, p = {p_fisher:.4f} {'(significant)' if p_fisher < 0.05 else '(not significant)'}")

In [ ]:
# Spearman correlation: confidence vs correctness
corr, p_corr = stats.spearmanr(df_subs["confidence"], df_subs["correct"])
print("Spearman correlation: confidence vs correctness (all annotations)")
print(f"  rho = {corr:.4f}, p = {p_corr:.4f} {'(significant)' if p_corr < 0.05 else '(not significant)'}")

print()

# Kruskal-Wallis: does accuracy differ across confidence levels (1, 2, 3)?
g1 = df_subs[df_subs["confidence"] == 1]["correct"].values
g2 = df_subs[df_subs["confidence"] == 2]["correct"].values
g3 = df_subs[df_subs["confidence"] == 3]["correct"].values
stat_kw, p_kw = stats.kruskal(g1, g2, g3)
print("Kruskal-Wallis: accuracy across confidence levels")
print(f"  H = {stat_kw:.4f}, p = {p_kw:.4f} {'(significant)' if p_kw < 0.05 else '(not significant)'}")

In [ ]:
# Per-subcategory: binomial test for accuracy on each sub-category
print("Binomial test: accuracy on sexist texts by sub-category (vs 50%)")
print(f"{'Sub-category':<20} {'Acc':>5} {'n':>3} {'p-value':>10} {'Sig?':>5}")
print("-" * 50)
for sub in subcategories:
    sub_data = df_subs[df_subs[sub] > 0]
    successes = sub_data["correct"].sum()
    n = len(sub_data)
    if n > 0:
        result = stats.binomtest(successes, n, p=0.5, alternative="two-sided")
        sig = "**" if result.pvalue < 0.01 else "*" if result.pvalue < 0.05 else "ns"
        print(f"{sub:<20} {successes/n:5.1%} {n:3d} {result.pvalue:10.4f} {sig:>5}")

print()

# Pairwise Fisher's exact: compare sub-category pairs
print("Pairwise Fisher's exact on sub-category detection rates (Bonferroni corrected):")
sub_pairs = list(combinations(subcategories, 2))
n_tests = len(sub_pairs)
for sub_a, sub_b in sub_pairs:
    for sub, label in [(sub_a, sub_a), (sub_b, sub_b)]:
        pass
    acc_a = df_subs[df_subs[sub_a] > 0]["correct"].values
    acc_b = df_subs[df_subs[sub_b] > 0]["correct"].values
    if len(acc_a) > 0 and len(acc_b) > 0:
        n_errors_a = int((acc_a == 0).sum())
        n_total_a = len(acc_a)
        n_errors_b = int((acc_b == 0).sum())
        n_total_b = len(acc_b)
        table = np.array([[n_errors_a, n_total_a - n_errors_a], [n_errors_b, n_total_b - n_errors_b]])
        _, p_raw = stats.fisher_exact(table)
        p_adj = min(p_raw * n_tests, 1.0)
        if p_raw < 0.05:
            sig = "**" if p_adj < 0.01 else "*" if p_adj < 0.05 else "ns (adj)"
            print(f"  {sub_a} vs {sub_b}: p_raw={p_raw:.4f}, p_adj={p_adj:.4f} {sig}")
print("  (Only showing pairs with raw p < .05)")